In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# CONFIGURATION & PATHS
# ============================================================
UCI_DIR      = "datasets/cuff+less+blood+pressure+estimation"
HOSP_META    = "datasets/5459299/Data File/PPG-BP dataset.xlsx"
HOSP_SIG_DIR = "datasets/5459299/Data File/0_subject"

PHASE_OUT_DIR = "outputs/phase_0"
DATA_OUT_DIR  = os.path.join(PHASE_OUT_DIR, "data")


MAX_LEN = 2100
PARTS   = ["Part_1.mat", "Part_2.mat", "Part_3.mat", "Part_4.mat"]

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 12

def setup_directories():
    """Create all required output directories."""
    os.makedirs(DATA_OUT_DIR, exist_ok=True)

def fix_len(sig, n=MAX_LEN):
    sig = np.asarray(sig, dtype=np.float32)
    if len(sig) >= n:
        return sig[:n]
    return np.pad(sig, (0, n - len(sig)), mode="edge")

# ============================================================
# TASK 1 — UCI Extraction
# ============================================================
def extract_uci():
    print("\n" + "="*55)
    print("TASK 1: UCI Cuffless BP — All 4 Parts")
    print("="*55)

    all_ppg, all_abp, all_ecg = [], [], []
    all_sbp, all_dbp          = [], []
    meta_rows                 = []

    for part_name in PARTS:
        fpath = os.path.join(UCI_DIR, part_name)
        if not os.path.exists(fpath):
            print(f"  [SKIP] {part_name} not found")
            continue

        print(f"\n  Loading {part_name} ...")

        with h5py.File(fpath, "r") as f:
            key  = next(k for k in f.keys() if not k.startswith("#"))
            refs = f[key]          
            n    = refs.shape[0]   
            ok   = 0

            for i in tqdm(range(n), desc=f"    {part_name}", ncols=70):
                try:
                    rec = np.array(f[refs[i, 0]])   # shape (T, 3)

                    ppg_raw = rec[:, 0].astype(np.float32)
                    abp_raw = rec[:, 1].astype(np.float32)
                    ecg_raw = rec[:, 2].astype(np.float32)

                    if len(ppg_raw) < 100:
                        continue

                    sbp = float(np.max(abp_raw))
                    dbp = float(np.min(abp_raw))

                    all_ppg.append(fix_len(ppg_raw))
                    all_abp.append(fix_len(abp_raw))
                    all_ecg.append(fix_len(ecg_raw))
                    all_sbp.append(sbp)
                    all_dbp.append(dbp)
                    meta_rows.append({
                        "part"       : part_name,
                        "record_idx" : i,
                        "sig_len_raw": len(ppg_raw),
                        "sbp"        : round(sbp, 3),
                        "dbp"        : round(dbp, 3),
                    })
                    ok += 1
                except Exception:
                    continue

            print(f"    → {ok} / {n} records extracted")

    if not all_ppg:
        print("  [ERROR] No records extracted for UCI.")
        return False

    X = np.stack([
        np.stack(all_ppg),
        np.stack(all_abp),
        np.stack(all_ecg),
    ], axis=1).astype(np.float32)

    y = np.column_stack([all_sbp, all_dbp]).astype(np.float32)

    np.savez_compressed(os.path.join(DATA_OUT_DIR, "uci_all_parts.npz"), X=X, y=y, channel_names=["PPG", "ABP", "ECG"])
    pd.DataFrame(meta_rows).to_csv(os.path.join(DATA_OUT_DIR, "uci_manifest.csv"), index=False)

    print(f"\n  Saved: {DATA_OUT_DIR}/uci_all_parts.npz")
    print(f"  X : {X.shape}   y : {y.shape}")
    print(f"  SBP  {y[:,0].mean():.1f} ± {y[:,0].std():.1f}  [{y[:,0].min():.0f} – {y[:,0].max():.0f}] mmHg")
    print(f"  DBP  {y[:,1].mean():.1f} ± {y[:,1].std():.1f}  [{y[:,1].min():.0f} – {y[:,1].max():.0f}] mmHg")
    return True

# ============================================================
# TASK 2 — Hospital Extraction
# ============================================================
def extract_hospital():
    print("\n" + "="*55)
    print("TASK 2: Hospital PPG-BP — Signal + Metadata Match")
    print("="*55)

    if not os.path.exists(HOSP_META):
        print(f"  [SKIP] {HOSP_META} not found")
        return False

    df_meta = pd.read_excel(HOSP_META, skiprows=1)
    df_meta.columns = [c.strip() for c in df_meta.columns]
    df_meta = df_meta.dropna(subset=["subject_ID"])
    df_meta["subject_ID"] = df_meta["subject_ID"].astype(int)
    print(f"  Metadata loaded: {len(df_meta)} subjects")

    signals, rows = [], []

    for _, meta_row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="  Matching", ncols=70):
        sid = int(meta_row["subject_ID"])

        for seg in [1, 2, 3]:
            fpath = os.path.join(HOSP_SIG_DIR, f"{sid}_{seg}.txt")
            if not os.path.exists(fpath):
                continue
            try:
                raw = np.loadtxt(fpath, dtype=np.float32)
            except Exception:
                continue

            signals.append(fix_len(raw))
            row = {"subject_ID": sid, "segment": seg, "sig_len_raw": len(raw)}
            for col in df_meta.columns:
                if col != "subject_ID":
                    row[col] = meta_row[col]
            rows.append(row)

    if not signals:
        print("  [ERROR] No signals matched for Hospital dataset.")
        return False

    X = np.stack(signals).astype(np.float32)
    
    np.savez_compressed(os.path.join(DATA_OUT_DIR, "hospital_matched.npz"), X=X)
    pd.DataFrame(rows).to_csv(os.path.join(DATA_OUT_DIR, "hospital_matched.csv"), index=False)

    print(f"\n  Saved: {DATA_OUT_DIR}/hospital_matched.npz   X:{X.shape}")
    print(f"  Saved: {DATA_OUT_DIR}/hospital_matched.csv   {len(rows)} rows")
    return True

# ============================================================
# MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    setup_directories()
    
    extract_uci()
    extract_hospital()
    
    print("\n=====================================================")
    print(" Phase 0 pipeline completed successfully!")
    print(f" Data saved in:  {DATA_OUT_DIR}")
    print("=====================================================")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import resample

# ============================================================
# 0. CONFIGURATION & STYLE
# ============================================================
TARGET_LEN = 1000
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3
})
C = {"ppg": "#2ca02c", "abp": "#d62728", "ecg": "#1f77b4", "hosp": "#ff7f0e"}

# ============================================================
# 1. CORE UTILITIES
# ============================================================
def preprocess_signal(sig, target_len=TARGET_LEN):
    """Resample and apply Min-Max Normalization [0, 1]"""
    sig = np.asarray(sig, dtype=np.float32)
    res = resample(sig, target_len)
    # Physics-ready scaling
    s_min, s_max = res.min(), res.max()
    if s_max - s_min < 1e-8: return np.zeros_like(res)
    return (res - s_min) / (s_max - s_min)

def get_htn_category(sbp, dbp):
    """Categorize BP according to AHA/ACC 2017"""
    if sbp >= 140 or dbp >= 90: return "Stage 2"
    if sbp >= 130 or dbp >= 80: return "Stage 1"
    if sbp >= 120 and dbp < 80: return "Elevated"
    return "Normal"

# ============================================================
# 2. DATA LOADING & PROCESSING
# ============================================================
print("[INFO] Loading and Preprocessing Tensors...")
uci = np.load("outputs/phase_0/data/uci_all_parts.npz")
hosp = np.load("outputs/phase_0/data/hospital_matched.npz")

X_uci, y_uci = uci["X"], uci["y"]
X_hosp = hosp["X"]

# Prepare single samples for visualization
uci_sample_ppg = preprocess_signal(X_uci[0, 0])
uci_sample_abp = preprocess_signal(X_uci[0, 1])
hosp_sample_ppg = preprocess_signal(X_hosp[0])

# ============================================================
# 3. VISUALIZATION: THE PERFECT PLOTS
# ============================================================

# --- A) Aligned Morphological Comparison ---
plt.figure(figsize=(10, 4))
plt.plot(uci_sample_ppg, color=C["ppg"], lw=1.5, label="UCI PPG (Reference)")
plt.plot(hosp_sample_ppg, color=C["hosp"], lw=1.5, ls="--", label="Hospital PPG (Target)")
plt.fill_between(range(TARGET_LEN), uci_sample_ppg, hosp_sample_ppg, color='gray', alpha=0.1)
plt.title("Morphological Comparison: UCI vs Hospital (Normalized)", fontweight="bold")
plt.xlabel("Normalized Time (Steps)")
plt.ylabel("Normalized Amplitude [0, 1]")
plt.legend(frameon=True, facecolor="white")
plt.tight_layout()
plt.show()

# --- B) PINN Multi-Channel View ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(uci_sample_ppg, color=C["ppg"], label="Input: PPG")
ax1.set_ylabel("PPG")
ax1.legend(loc="upper right")
ax1.set_title("Multi-Modal Reference (UCI Bundle)", fontweight="bold")

ax2.plot(uci_sample_abp, color=C["abp"], label="Ground Truth: ABP")
ax2.set_ylabel("ABP")
ax2.legend(loc="upper right")
plt.xlabel("Time Steps")
plt.tight_layout()
plt.show()

# --- C) Clinical Distribution (BP Classes) ---
sbp, dbp = y_uci[:, 0], y_uci[:, 1]
categories = [get_htn_category(s, d) for s, d in zip(sbp, dbp)]
cat_counts = pd.Series(categories).value_counts()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(sbp, bins=40, color=C["abp"], kde=True)
plt.axvline(130, color="black", ls="--", alpha=0.6, label="HTN Threshold")
plt.title("SBP Distribution (UCI)")
plt.xlabel("mmHg")

plt.subplot(1, 2, 2)
cat_counts.plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette("viridis"), startangle=140)
plt.title("Dataset BP Composition")
plt.ylabel("")
plt.tight_layout()
plt.show()

# ============================================================
# 4. ENHANCED SUMMARY
# ============================================================
# Batch STD calculation with vectorized approach
uci_all_ppg = X_uci[:, 0, :] # Raw signals
uci_std_dist = np.std(uci_all_ppg, axis=1)
hosp_std_dist = np.std(X_hosp, axis=1)

print("\n" + "="*40)
print(f"{'CLINICAL SUMMARY':^40}")
print("="*40)
print(f"UCI SBP:      {sbp.mean():.2f} ± {sbp.std():.2f} mmHg")
print(f"UCI DBP:      {dbp.mean():.2f} ± {dbp.std():.2f} mmHg")
print(f"Prevalence of Stage 2 HTN: {(cat_counts.get('Stage 2', 0)/len(sbp))*100:.1f}%")
print("-" * 40)
print(f"UCI Signal Variability:      {uci_std_dist.mean():.4f}")
print(f"Hospital Signal Variability: {hosp_std_dist.mean():.4f}")
print("="*40)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 0. CONFIGURATION & STYLE (Research Standard)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "figure.dpi": 200,
    "axes.grid": False,
    "savefig.bbox": "tight"
})
palette = sns.color_palette("viridis", as_cmap=False)

# ============================================================
# 1. LOAD & CLEAN DATA
# ============================================================
hosp = np.load("outputs/phase_0/data/hospital_matched.npz")
df = pd.read_csv("outputs/phase_0/data/hospital_matched.csv")

# Rename for clarity
df = df.rename(columns={
    "Systolic Blood Pressure(mmHg)": "SBP",
    "Diastolic Blood Pressure(mmHg)": "DBP",
    "Age(year)": "Age",
    "Sex(M/F)": "Sex"
})

# 1.1 Clinical Label Mapping (From Hospital records)
def clean_hosp_label(x):
    if pd.isna(x): return "Unknown"
    x = str(x).lower()
    if "stage 2" in x: return "Stage 2 HTN"
    if "stage 1" in x: return "Stage 1 HTN"
    if "pre" in x:     return "Pre-HTN"
    if "normal" in x:  return "Normal"
    return "Other"

df["hosp_stage"] = df["Hypertension"].apply(clean_hosp_label)

# 1.2 Unified Rule-based Labeling (AHA/ACC 2017 Guidelines)
def get_bp_category(sbp, dbp):
    if sbp >= 140 or dbp >= 90: return "Stage 2 HTN"
    if sbp >= 130 or dbp >= 80: return "Stage 1 HTN"
    if sbp >= 120 and dbp < 80: return "Pre-HTN"
    return "Normal"

df["rule_stage"] = df.apply(lambda r: get_bp_category(r["SBP"], r["DBP"]), axis=1)

# Ensure categorical order for plotting
cat_order = ["Normal", "Pre-HTN", "Stage 1 HTN", "Stage 2 HTN"]
df["hosp_stage"] = pd.Categorical(df["hosp_stage"], categories=cat_order + ["Other", "Unknown"], ordered=True)
df["rule_stage"] = pd.Categorical(df["rule_stage"], categories=cat_order, ordered=True)

# ============================================================
# 2. ANALYSIS & TABLES
# ============================================================
print("\n" + "="*40)
print(f"{'LABEL AGREEMENT SUMMARY':^40}")
print("="*40)
print(df[["hosp_stage", "rule_stage"]].value_counts().sort_index())

# ============================================================
# 3. VISUALIZATION (Publication Quality)
# ============================================================

# --- 3.1 Distribution Comparison ---
fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

sns.countplot(data=df, x="hosp_stage", ax=ax[0], palette="Blues_d")
ax[0].set_title("Original Hospital Labels", fontweight="bold")
ax[0].tick_params(axis='x', rotation=45)

sns.countplot(data=df, x="rule_stage", ax=ax[1], palette="Reds_d")
ax[1].set_title("Rule-based (AHA/ACC 2017) Labels", fontweight="bold")
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# --- 3.2 Confusion Matrix / Agreement Heatmap ---
plt.figure(figsize=(10, 7))
ct = pd.crosstab(df["hosp_stage"], df["rule_stage"])

sns.heatmap(ct, annot=True, fmt="d", cmap="YlGnBu", cbar_kws={'label': 'Number of Subjects'})
plt.title("Agreement Matrix: Hospital Label vs. Clinical Rule", fontweight="bold", pad=20)
plt.xlabel("Guideline-based Category (Rule)")
plt.ylabel("Original Record Category (Hosp)")
plt.tight_layout()
plt.show()

# ============================================================
# 4. SIGNAL QUALITY ANALYSIS (Physics-Informed)
# ============================================================
X = hosp["X"]
df["sig_std"] = X.std(axis=1)
df["sig_mean"] = X.mean(axis=1)
df["cv"] = (df["sig_std"] / (df["sig_mean"] + 1e-8)) * 100  # Coefficient of Variation

print("\n" + "="*40)
print(f"{'SIGNAL QUALITY BY CATEGORY':^40}")
print("="*40)
quality_stats = df.groupby("hosp_stage", observed=True).agg({
    "sig_std": "mean",
    "cv": "mean"
}).rename(columns={"sig_std": "Mean_STD", "cv": "Mean_CV(%)"})
print(quality_stats)

# --- 4.1 Boxplot Quality Analysis ---
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="hosp_stage", y="sig_std", palette="Set2")
plt.title("Signal Variability (STD) Across Clinical Stages", fontweight="bold")
plt.ylabel("Amplitude STD")
plt.show()